# 🛰️ Sentinel-2 Water Segmentation: Flask REST API & Web UI on Kaggle
This is a **complete, self-contained notebook** to run and serve the 12-channel Sentinel-2 Water Segmentation model (ImageNet-pretrained ResNet34 U-Net) as a Flask web service directly from Kaggle.

---

### ⚙️ Kaggle Settings Checklist
Before running the notebook, verify two settings in the right-hand panel (**Notebook options**):
1. **Internet**: Toggle **ON** (Required to download packages and initialize the Cloudflare tunnel).
2. **Accelerator**: **GPU (T4 x2 or P100)** is recommended for ~10–20 ms inference (CPU also works).

---

### 🌟 What this notebook provides:
1. **Zero External Clones**: Writes all application source code, HTML/CSS/JS frontend, and normalisation configurations in-place.
2. **Automatic Weight Detection**: Automatically scans `/kaggle/input/` and `/kaggle/working/` for `pretrained_resnet34_u_net_12ch_best.pt`.
3. **Instant Demo Mode**: If you haven't attached your weights yet, it automatically generates a temporary architecture-matched model so you can test the UI and REST API immediately without crashing.
4. **Zero-Config Public Tunnel**: Exposes the Flask app using **Cloudflare Tunnel (`cloudflared`)** with a free, secure `https://*.trycloudflare.com` URL (no account or login required).
5. **Interactive In-Notebook UI**: Embeds the full drag-and-drop web app directly inside the notebook.
6. **In-Notebook Python Client & Visualization**: Synthesizes a 12-channel satellite tile, calls `/predict`, and renders the true-color RGB preview, predicted mask, and blue water overlay using Matplotlib.
7. **Automated Unit Tests**: Includes the full 25-test suite for API plumbing, formats, and preprocessing verification.


In [ ]:
# ==============================================================================
# Step 1: Install Dependencies & Check Hardware
# ==============================================================================
!pip install -q segmentation-models-pytorch==0.5.0 tifffile flask pyngrok

import os
import sys
import json
import time
import subprocess
import urllib.request
import numpy as np
import torch

print("=" * 60)
print(f"Python version:  {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU device:      {torch.cuda.get_device_name(0)} (CUDA is available)")
else:
    print("Device:          CPU (CUDA is not available)")
print("=" * 60)


---
## 📦 Step 2: Generate Application Code & Web Assets

We now generate the complete application files inside `water_app/`:
```text
water_app/
  ├── app.py                # Flask REST API endpoints (/predict, /health, /)
  ├── inference.py          # WaterSegmenter inference class & preprocessing
  ├── export_model.py       # Checkpoint sanitization and bundle packaging
  ├── static/
  │   └── index.html        # Interactive web UI (drag & drop, threshold slider)
  ├── tests/
  │   └── test_api.py       # 25 automated unit tests
  └── model_bundle/
      ├── preprocessing.json # Normalization parameters & architecture specs
      └── water_unet_resnet34_12ch.pt # Model weights
```


In [ ]:
# Create application folder structure
if os.path.exists("/kaggle/working"):
    os.chdir("/kaggle/working")

APP_DIR = os.path.abspath("water_app")
STATIC_DIR = os.path.join(APP_DIR, "static")
TESTS_DIR = os.path.join(APP_DIR, "tests")
BUNDLE_DIR = os.path.join(APP_DIR, "model_bundle")

os.makedirs(STATIC_DIR, exist_ok=True)
os.makedirs(TESTS_DIR, exist_ok=True)
os.makedirs(BUNDLE_DIR, exist_ok=True)
print(f"Application directory initialized at: {APP_DIR}")


In [ ]:
%%writefile water_app/static/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Water Segmentation</title>
<style>
  :root {
    --bg: #f6f7f9; --card: #ffffff; --text: #1b2330; --muted: #5d6b7e; --line: #dde2ea;
    --accent: #1f6feb; --accent-text: #ffffff; --danger: #b42318; --danger-bg: #fdeceb;
  }
  @media (prefers-color-scheme: dark) {
    :root {
      --bg: #0f141b; --card: #171e28; --text: #e6ebf2; --muted: #95a3b8; --line: #2a3442;
      --accent: #4c8dff; --accent-text: #0b1220; --danger: #ffb4ab; --danger-bg: #3a1b19;
    }
  }
  * { box-sizing: border-box; }
  body { margin: 0; background: var(--bg); color: var(--text);
         font: 16px/1.5 system-ui, -apple-system, "Segoe UI", Roboto, sans-serif; }
  main { max-width: 960px; margin: 0 auto; padding: 24px 16px 48px; }
  h1 { font-size: 1.5rem; margin: 0 0 4px; }
  .sub { color: var(--muted); margin: 0 0 20px; }
  .card { background: var(--card); border: 1px solid var(--line); border-radius: 12px; padding: 16px; }
  #drop { border: 2px dashed var(--line); border-radius: 12px; padding: 28px 16px; text-align: center;
          color: var(--muted); cursor: pointer; transition: border-color .15s, background .15s; }
  #drop.over, #drop:focus-visible { border-color: var(--accent); outline: none; }
  #drop strong { color: var(--text); }
  .row { display: flex; flex-wrap: wrap; gap: 12px 20px; align-items: center; margin-top: 14px; }
  label { color: var(--muted); font-size: .9rem; }
  input[type=range] { vertical-align: middle; width: 160px; accent-color: var(--accent); }
  button { background: var(--accent); color: var(--accent-text); border: 0; border-radius: 8px;
           padding: 9px 18px; font: inherit; font-weight: 600; cursor: pointer; }
  button:disabled { opacity: .5; cursor: not-allowed; }
  #error { display: none; margin-top: 14px; padding: 10px 14px; border-radius: 8px;
           background: var(--danger-bg); color: var(--danger); }
  #results { display: none; margin-top: 20px; }
  .grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; }
  figure { margin: 0; text-align: center; }
  figure img { width: 100%; image-rendering: pixelated; border-radius: 8px; border: 1px solid var(--line);
               background: #000; display: block; }
  figcaption { color: var(--muted); font-size: .85rem; margin-top: 6px; }
  .stats { display: flex; flex-wrap: wrap; gap: 8px 28px; margin-top: 16px; }
  .stat b { display: block; font-size: 1.25rem; }
  .stat span { color: var(--muted); font-size: .85rem; }
  .legend i { display: inline-block; width: 12px; height: 12px; border-radius: 3px; vertical-align: -1px; margin-right: 4px; }
  @media (max-width: 640px) { .grid { grid-template-columns: 1fr; } }
</style>
</head>
<body>
<main>
  <h1>Water segmentation</h1>
  <p class="sub">Upload a 12-band satellite tile (.tif / .tiff / .npy) and get a per-pixel water mask.</p>

  <section class="card">
    <div id="drop" tabindex="0" role="button" aria-label="Choose a file to upload">
      <p><strong>Drop a file here</strong> or click to choose one</p>
      <p id="fname">No file selected</p>
    </div>
    <input id="file" type="file" accept=".tif,.tiff,.npy" hidden>

    <div class="row">
      <label for="thr">Threshold: <b id="thrv">0.50</b></label>
      <input id="thr" type="range" min="0.05" max="0.95" step="0.05" value="0.5">
      <button id="run" disabled>Segment</button>
    </div>
    <div id="error" role="alert"></div>
  </section>

  <section id="results" class="card" aria-live="polite">
    <div class="grid">
      <figure><img id="rgb" alt="True-colour preview of the uploaded tile"><figcaption>True colour (bands 4, 3, 2)</figcaption></figure>
      <figure><img id="mask" alt="Predicted water mask"><figcaption>Predicted mask (white = water)</figcaption></figure>
      <figure><img id="overlay" alt="Mask overlaid on the tile"><figcaption class="legend"><i style="background:#1e6eff"></i>Water overlay</figcaption></figure>
    </div>
    <div class="stats">
      <div class="stat"><b id="frac">-</b><span>water coverage</span></div>
      <div class="stat"><b id="size">-</b><span>tile size (px)</span></div>
      <div class="stat"><b id="ms">-</b><span>inference time</span></div>
    </div>
  </section>
</main>

<script>
(() => {
  const $ = id => document.getElementById(id);
  const drop = $('drop'), fileInput = $('file'), run = $('run'), err = $('error');
  let file = null;

  const setFile = f => {
    file = f || null;
    $('fname').textContent = file ? file.name : 'No file selected';
    run.disabled = !file;
    err.style.display = 'none';
  };
  const showError = msg => { err.textContent = msg; err.style.display = 'block'; };

  drop.addEventListener('click', () => fileInput.click());
  drop.addEventListener('keydown', e => { if (e.key === 'Enter' || e.key === ' ') { e.preventDefault(); fileInput.click(); } });
  fileInput.addEventListener('change', () => setFile(fileInput.files[0]));
  ['dragenter', 'dragover'].forEach(t => drop.addEventListener(t, e => { e.preventDefault(); drop.classList.add('over'); }));
  ['dragleave', 'drop'].forEach(t => drop.addEventListener(t, e => { e.preventDefault(); drop.classList.remove('over'); }));
  drop.addEventListener('drop', e => setFile(e.dataTransfer.files[0]));

  $('thr').addEventListener('input', e => { $('thrv').textContent = (+e.target.value).toFixed(2); });

  async function segment() {
    if (!file) return;
    run.disabled = true; run.textContent = 'Working...'; err.style.display = 'none';
    const fd = new FormData();
    fd.append('image', file);
    fd.append('threshold', $('thr').value);
    try {
      const res = await fetch('/predict', { method: 'POST', body: fd });
      const data = await res.json().catch(() => ({ error: 'The server returned an unreadable response.' }));
      if (!res.ok) throw new Error(data.error || ('Request failed (' + res.status + ')'));
      $('rgb').src = 'data:image/png;base64,' + data.rgb_png_base64;
      $('mask').src = 'data:image/png;base64,' + data.mask_png_base64;
      $('overlay').src = 'data:image/png;base64,' + data.overlay_png_base64;
      $('frac').textContent = (data.water_fraction * 100).toFixed(1) + '%';
      $('size').textContent = data.width + ' x ' + data.height;
      $('ms').textContent = data.inference_ms + ' ms';
      $('results').style.display = 'block';
    } catch (e) {
      $('results').style.display = 'none';
      showError(e.message);
    } finally {
      run.disabled = !file; run.textContent = 'Segment';
    }
  }
  run.addEventListener('click', segment);
})();
</script>
</body>
</html>


In [ ]:
%%writefile water_app/inference.py
"""Inference for the Week 2 model: ImageNet-pretrained ResNet34 U-Net, 12 input channels.

Usage as a library:
    seg = WaterSegmenter("model_bundle")
    mask = seg.predict(raw_array)          # (H, W) uint8, 1 = water, 0 = not water

Usage from the command line:
    python inference.py tile.tif --bundle model_bundle --out mask.png

The input must be the RAW 12-band tile (same values as the .tif files in the training
dataset). All preprocessing (min/max scaling, z-score) is done in here, using the
statistics saved in the bundle, so the caller never has to normalise anything.
"""
import argparse
import io
import json
import os

import numpy as np
import torch
import torch.nn.functional as F
import segmentation_models_pytorch as smp
import tifffile
from PIL import Image

NUM_CHANNELS = 12
DIVISOR = 32          # a ResNet U-Net halves the image 5 times, so H and W must be multiples of 32
MAX_SIDE = 1024       # refuse absurdly large inputs (memory); training tiles are 128x128
MIN_SIDE = 8
RGB_BANDS = (3, 2, 1)  # Red, Green, Blue positions in the 12-band stack (used only for previews)
WEIGHTS_FILE = "water_unet_resnet34_12ch.pt"
PREPROC_FILE = "preprocessing.json"


class InputError(ValueError):
    """The uploaded data cannot be used (wrong shape, wrong type, NaN, ...). Maps to HTTP 400."""


# --------------------------------------------------------------------------------------
# Reading / validating input
# --------------------------------------------------------------------------------------
def to_chw(arr):
    """Return a channel-first (12, H, W) float32 array, or raise InputError.

    Same convention as the training notebook: if the first axis is 12 it is already (C,H,W);
    otherwise a last axis of 12 is treated as channels-last (H,W,C) and transposed.
    """
    arr = np.asarray(arr)
    if arr.ndim != 3:
        raise InputError(f"Expected a 3-D array with 12 channels, got shape {tuple(arr.shape)}.")
    if arr.shape[0] == NUM_CHANNELS:
        pass
    elif arr.shape[-1] == NUM_CHANNELS:
        arr = np.transpose(arr, (2, 0, 1))
    else:
        raise InputError(f"Expected 12 channels (shape (12,H,W) or (H,W,12)), got shape {tuple(arr.shape)}.")
    if not (np.issubdtype(arr.dtype, np.number) or np.issubdtype(arr.dtype, np.bool_)):
        raise InputError(f"Array dtype {arr.dtype} is not numeric.")
    arr = arr.astype(np.float32)
    h, w = arr.shape[1:]
    if min(h, w) < MIN_SIDE or max(h, w) > MAX_SIDE:
        raise InputError(f"Image size {h}x{w} is outside the supported range "
                         f"({MIN_SIDE}..{MAX_SIDE} pixels per side; training tiles are 128x128).")
    if not np.isfinite(arr).all():
        raise InputError("The image contains NaN or infinite values. Fill or remove them before uploading.")
    return arr


def read_array(data, filename):
    """Decode uploaded bytes (.tif/.tiff/.npy) into a raw array, or raise InputError."""
    ext = os.path.splitext(filename or "")[1].lower()
    try:
        if ext in (".tif", ".tiff"):
            return tifffile.imread(io.BytesIO(data))
        if ext == ".npy":
            return np.load(io.BytesIO(data), allow_pickle=False)   # never unpickle uploads
    except Exception as e:
        raise InputError(f"Could not read '{filename}' as a {ext} file: {type(e).__name__}.") from e
    raise InputError(f"Unsupported file type '{ext}'. Upload a 12-band .tif/.tiff or .npy file.")


def stretch(band, p_lo=2, p_hi=98):
    lo, hi = np.percentile(band, [p_lo, p_hi])
    return np.zeros_like(band) if hi <= lo else np.clip((band - lo) / (hi - lo), 0, 1)


def rgb_preview(raw_chw):
    """(H,W,3) uint8 true-colour preview from the raw bands (2-98 percentile stretch)."""
    rgb = np.stack([stretch(raw_chw[i]) for i in RGB_BANDS], axis=-1)
    return (rgb * 255).astype(np.uint8)


# --------------------------------------------------------------------------------------
# Model
# --------------------------------------------------------------------------------------
def build_model(encoder="resnet34", in_channels=NUM_CHANNELS):
    """The same architecture the notebook trained. smp builds the first conv with `in_channels`
    inputs directly; the weights are then overwritten by the checkpoint, so no ImageNet download
    (and no internet) is needed here."""
    return smp.Unet(encoder_name=encoder, encoder_weights=None, in_channels=in_channels, classes=1)


class WaterSegmenter:
    def __init__(self, bundle_dir, device=None):
        with open(os.path.join(bundle_dir, PREPROC_FILE)) as f:
            self.cfg = json.load(f)
        c = self.cfg
        for key in ("min", "max", "mean", "std"):
            if len(c[key]) != NUM_CHANNELS:
                raise ValueError(f"preprocessing.json: '{key}' must have {NUM_CHANNELS} values, has {len(c[key])}")
        self.ch_min = np.array(c["min"], dtype=np.float32).reshape(-1, 1, 1)
        self.ch_max = np.array(c["max"], dtype=np.float32).reshape(-1, 1, 1)
        self.mean = np.array(c["mean"], dtype=np.float32).reshape(-1, 1, 1)
        self.std = np.array(c["std"], dtype=np.float32).reshape(-1, 1, 1)
        self.threshold = float(c.get("threshold", 0.5))

        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.model = build_model(c.get("encoder", "resnet34"), NUM_CHANNELS)
        state = torch.load(os.path.join(bundle_dir, c.get("weights_file", WEIGHTS_FILE)),
                           map_location="cpu", weights_only=True)
        self.model.load_state_dict(state, strict=True)
        self.model.to(self.device).eval()

    # -- preprocessing: identical to the training notebook (min/max -> clip -> z-score) ----------
    def preprocess(self, raw_chw):
        """raw_chw: (12,H,W) RAW values -> standardised float32 network input."""
        x = np.clip((raw_chw - self.ch_min) / np.maximum(self.ch_max - self.ch_min, 1e-6), 0, 1)
        return ((x - self.mean) / self.std).astype(np.float32)

    @torch.no_grad()
    def predict_proba(self, raw):
        """raw: array-like in any accepted layout. Returns (H,W) float32 water probability."""
        raw = to_chw(raw)
        h, w = raw.shape[1:]
        x = torch.from_numpy(self.preprocess(raw)).unsqueeze(0).to(self.device)
        ph, pw = (-h) % DIVISOR, (-w) % DIVISOR
        if ph or pw:
            mode = "reflect" if (ph < h and pw < w) else "replicate"
            x = F.pad(x, (0, pw, 0, ph), mode=mode)
        prob = torch.sigmoid(self.model(x))[0, 0, :h, :w]
        return prob.float().cpu().numpy()

    def predict(self, raw, threshold=None):
        """Returns (H,W) uint8 mask: 1 = water, 0 = not water."""
        t = self.threshold if threshold is None else float(threshold)
        return (self.predict_proba(raw) > t).astype(np.uint8)


def mask_to_png_bytes(mask01):
    """0/1 mask -> 8-bit PNG with values 0 (not water) / 255 (water)."""
    buf = io.BytesIO()
    Image.fromarray((mask01 * 255).astype(np.uint8), mode="L").save(buf, format="PNG")
    return buf.getvalue()


def main():
    ap = argparse.ArgumentParser(description="Run water segmentation on one 12-band tile.")
    ap.add_argument("image", help="path to a 12-band .tif/.tiff/.npy tile")
    ap.add_argument("--bundle", default="model_bundle")
    ap.add_argument("--out", default="mask.png", help="output PNG path (0 = not water, 255 = water)")
    ap.add_argument("--threshold", type=float, default=None, help="probability threshold (default: from bundle)")
    args = ap.parse_args()

    with open(args.image, "rb") as f:
        raw = read_array(f.read(), args.image)
    seg = WaterSegmenter(args.bundle)
    mask = seg.predict(raw, args.threshold)
    with open(args.out, "wb") as f:
        f.write(mask_to_png_bytes(mask))
    print(f"{args.image}: {mask.shape[0]}x{mask.shape[1]} mask written to {args.out} "
          f"(water fraction {mask.mean():.3f})")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile water_app/export_model.py
"""Package the trained 12-channel pretrained ResNet34 U-Net into a self-contained deployment bundle.

Inputs (all produced by the Week 2 notebook, in its `week2_outputs/` folder):
    pretrained_resnet34_u_net_12ch.pt   the weights (a state_dict)
    channel_minmax.json                 per-channel min/max of the TRAIN set (12 values each)
    input_zscore_stats.json             per-channel mean/std of the final network input (16 values; the first 12 are used)

Output folder (default model_bundle/):
    water_unet_resnet34_12ch.pt         weights, verified to load into the inference architecture
    preprocessing.json                  stats + threshold + architecture info needed at inference time

Example:
    python export_model.py --checkpoint week2_outputs/pretrained_resnet34_u_net_12ch.pt \
        --minmax week2_outputs/channel_minmax.json --zstats week2_outputs/input_zscore_stats.json
"""
import argparse
import hashlib
import json
import os

import numpy as np
import torch
import segmentation_models_pytorch as smp

from inference import NUM_CHANNELS, WEIGHTS_FILE, PREPROC_FILE, build_model

BAND_NAMES = ['Coastal aerosol', 'Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2',
              'QA Band', 'Merit DEM', 'Copernicus DEM', 'ESA WorldCover', 'Water occurrence probability']


def load_state_dict(path):
    obj = torch.load(path, map_location="cpu", weights_only=True)
    for key in ("state_dict", "model_state_dict"):          # tolerate wrapped checkpoints
        if isinstance(obj, dict) and key in obj and isinstance(obj[key], dict):
            obj = obj[key]
    if not isinstance(obj, dict):
        raise SystemExit(f"{path} does not contain a state_dict.")
    return {k[len("module."):] if k.startswith("module.") else k: v for k, v in obj.items()}


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def export_bundle(checkpoint, minmax_path, zstats_path, out_dir, threshold=0.5, encoder="resnet34"):
    state = load_state_dict(checkpoint)

    # 1. Make sure this really is the 12-channel model.
    first = state.get("encoder.conv1.weight")
    if first is None:
        raise SystemExit("No 'encoder.conv1.weight' in the checkpoint - is this a ResNet U-Net checkpoint?")
    if first.shape[1] != NUM_CHANNELS:
        raise SystemExit(f"First conv takes {first.shape[1]} input channels, expected {NUM_CHANNELS}. "
                         f"Did you pick the 16-channel checkpoint by mistake?")

    # 2. Strict load into the inference architecture + smoke test.
    model = build_model(encoder, NUM_CHANNELS)
    model.load_state_dict(state, strict=True)
    model.eval()
    with torch.no_grad():
        out = model(torch.zeros(1, NUM_CHANNELS, 128, 128))
    assert out.shape == (1, 1, 128, 128) and torch.isfinite(out).all(), "smoke test forward pass failed"

    # 3. Preprocessing statistics.
    with open(minmax_path) as f:
        mm = json.load(f)
    with open(zstats_path) as f:
        zs = json.load(f)
    if len(mm["min"]) != NUM_CHANNELS or len(mm["max"]) != NUM_CHANNELS:
        raise SystemExit(f"channel_minmax.json must have {NUM_CHANNELS} values per key.")
    if len(zs["mean"]) < NUM_CHANNELS:
        raise SystemExit(f"input_zscore_stats.json has only {len(zs['mean'])} channels, need >= {NUM_CHANNELS}.")
    # The z-score stats were computed on the 16-channel input; channels 0..11 are exactly the 12 min/max-scaled
    # bands, so the first 12 entries are also the correct stats for the 12-channel model.

    os.makedirs(out_dir, exist_ok=True)
    weights_path = os.path.join(out_dir, WEIGHTS_FILE)
    torch.save(model.state_dict(), weights_path)

    cfg = {
        "architecture": "Unet",
        "encoder": encoder,
        "in_channels": NUM_CHANNELS,
        "weights_file": WEIGHTS_FILE,
        "weights_sha256": sha256(weights_path),
        "smp_version": smp.__version__,
        "torch_version": torch.__version__,
        "threshold": threshold,
        "band_names": BAND_NAMES,
        "min": [float(v) for v in mm["min"]],
        "max": [float(v) for v in mm["max"]],
        "mean": [float(v) for v in zs["mean"][:NUM_CHANNELS]],
        "std": [float(v) for v in zs["std"][:NUM_CHANNELS]],
        "preprocessing": "x = clip((raw - min) / (max - min), 0, 1); x = (x - mean) / std",
    }
    with open(os.path.join(out_dir, PREPROC_FILE), "w") as f:
        json.dump(cfg, f, indent=2)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Exported to {out_dir}/  ({n_params/1e6:.2f}M parameters, "
          f"{os.path.getsize(weights_path)/1e6:.1f} MB, sha256 {cfg['weights_sha256'][:12]}...)")
    return cfg


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--checkpoint", required=True)
    ap.add_argument("--minmax", required=True)
    ap.add_argument("--zstats", required=True)
    ap.add_argument("--out", default="model_bundle")
    ap.add_argument("--threshold", type=float, default=0.5)
    ap.add_argument("--encoder", default="resnet34")
    a = ap.parse_args()
    export_bundle(a.checkpoint, a.minmax, a.zstats, a.out, a.threshold, a.encoder)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile water_app/app.py
"""Flask REST API for water segmentation (12-channel pretrained ResNet34 U-Net).

Run (development):
    python app.py --bundle model_bundle            # http://127.0.0.1:5000
Run (production-style, one worker because the model lives in memory):
    MODEL_DIR=model_bundle gunicorn -w 1 --threads 4 -b 0.0.0.0:5000 "app:create_app()"

Endpoints
    GET  /          the HTML frontend
    GET  /health    model + service info (JSON)
    POST /predict   multipart/form-data, file field "image" (.tif/.tiff/.npy, 12 bands)
                    optional field/query "threshold" in [0,1]
                    ?format=json (default): JSON with the mask, a preview and statistics
                    ?format=png           : the raw mask as an 8-bit PNG (0 = not water, 255 = water)
"""
import argparse
import base64
import io
import logging
import os
import threading
import time

import numpy as np
from flask import Flask, jsonify, request, send_file, send_from_directory
from PIL import Image
from werkzeug.exceptions import HTTPException

from inference import InputError, WaterSegmenter, mask_to_png_bytes, read_array, rgb_preview, to_chw

log = logging.getLogger("water_api")
MAX_UPLOAD_BYTES = 32 * 1024 * 1024
HERE = os.path.dirname(os.path.abspath(__file__))


def _png_b64(arr, mode):
    buf = io.BytesIO()
    Image.fromarray(arr, mode=mode).save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("ascii")


def _overlay(rgb, mask01):
    """Blend a semi-transparent blue over the water pixels of the RGB preview."""
    out = rgb.astype(np.float32)
    water = mask01.astype(bool)
    out[water] = 0.45 * out[water] + 0.55 * np.array([30, 110, 255], dtype=np.float32)
    return out.astype(np.uint8)


def create_app(bundle_dir=None):
    bundle_dir = bundle_dir or os.environ.get("MODEL_DIR", "model_bundle")
    segmenter = WaterSegmenter(bundle_dir)
    lock = threading.Lock()        # one forward pass at a time; keeps memory use predictable

    app = Flask(__name__, static_folder=None)
    app.config["MAX_CONTENT_LENGTH"] = MAX_UPLOAD_BYTES
    app.config["SEGMENTER"] = segmenter

    # ---------------------------------------------------------------- errors -> JSON
    @app.errorhandler(InputError)
    def _input_error(e):
        return jsonify(error=str(e)), 400

    @app.errorhandler(HTTPException)
    def _http_error(e):
        msg = "Upload too large (limit %d MB)." % (MAX_UPLOAD_BYTES // 2**20) if e.code == 413 else e.description
        return jsonify(error=msg), e.code

    @app.errorhandler(Exception)
    def _unexpected(e):
        log.exception("Unhandled error")
        return jsonify(error="Internal server error."), 500

    # ---------------------------------------------------------------- routes
    @app.get("/")
    def index():
        return send_from_directory(os.path.join(HERE, "static"), "index.html")

    @app.get("/health")
    def health():
        c = segmenter.cfg
        return jsonify(status="ok", architecture=c.get("architecture"), encoder=c.get("encoder"),
                       in_channels=c.get("in_channels"), threshold=segmenter.threshold,
                       device=str(segmenter.device), weights_sha256=c.get("weights_sha256"))

    @app.post("/predict")
    def predict():
        f = request.files.get("image")
        if f is None or not f.filename:
            raise InputError("No file uploaded. Send multipart/form-data with a file in the field 'image'.")

        thr_raw = request.form.get("threshold", request.args.get("threshold"))
        threshold = None
        if thr_raw not in (None, ""):
            try:
                threshold = float(thr_raw)
            except ValueError:
                raise InputError("'threshold' must be a number between 0 and 1.")
            if not (0.0 <= threshold <= 1.0):
                raise InputError("'threshold' must be between 0 and 1.")

        fmt = request.args.get("format", "json").lower()
        if fmt not in ("json", "png"):
            raise InputError("'format' must be 'json' or 'png'.")

        raw = to_chw(read_array(f.read(), f.filename))
        t0 = time.perf_counter()
        with lock:
            mask = segmenter.predict(raw, threshold)
        ms = (time.perf_counter() - t0) * 1000
        used_thr = segmenter.threshold if threshold is None else threshold
        frac = float(mask.mean())

        if fmt == "png":
            resp = send_file(io.BytesIO(mask_to_png_bytes(mask)), mimetype="image/png")
            resp.headers["X-Water-Fraction"] = f"{frac:.6f}"
            resp.headers["X-Threshold"] = f"{used_thr}"
            return resp

        rgb = rgb_preview(raw)
        return jsonify(
            height=int(mask.shape[0]), width=int(mask.shape[1]),
            threshold=used_thr,
            water_pixels=int(mask.sum()), water_fraction=frac,
            inference_ms=round(ms, 1),
            mask_png_base64=_png_b64(mask * 255, "L"),
            rgb_png_base64=_png_b64(rgb, "RGB"),
            overlay_png_base64=_png_b64(_overlay(rgb, mask), "RGB"),
        )

    return app


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--bundle", default=os.environ.get("MODEL_DIR", "model_bundle"))
    ap.add_argument("--host", default="127.0.0.1", help="use 0.0.0.0 only if you want other machines to reach it")
    ap.add_argument("--port", type=int, default=5000)
    args = ap.parse_args()
    logging.basicConfig(level=logging.INFO)
    create_app(args.bundle).run(host=args.host, port=args.port, threaded=True)


In [ ]:
%%writefile water_app/tests/test_api.py
"""Tests for export_model.py, inference.py and app.py.

They use a RANDOMLY initialised network with made-up normalisation stats, so they check the plumbing
(shapes, formats, error handling, preprocessing, checkpoint compatibility) - not segmentation quality.
Quality is checked on real tiles with evaluate_deployed.py.

Run from the project folder:   python -m unittest discover -s tests -v
"""
import base64
import io
import json
import os
import sys
import tempfile
import unittest

import numpy as np
import tifffile
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from PIL import Image

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from app import create_app                      # noqa: E402
from export_model import export_bundle          # noqa: E402
from inference import WaterSegmenter, to_chw    # noqa: E402

RNG = np.random.default_rng(0)
MIN = np.zeros(12); MAX = np.full(12, 10000.0)


def notebook_style_checkpoint(path):
    """Build the model exactly like the notebook does (3-channel smp model, first conv swapped for a
    12-channel one) and save its state_dict - proves the real checkpoint format loads at inference."""
    model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1)
    old = next(m for m in model.encoder.modules() if isinstance(m, nn.Conv2d) and m.in_channels == 3)
    new = nn.Conv2d(12, old.out_channels, old.kernel_size, old.stride, old.padding, bias=old.bias is not None)
    nn.init.kaiming_normal_(new.weight, mode="fan_out", nonlinearity="relu")
    model.encoder.conv1 = new
    torch.save(model.state_dict(), path)


def tif_bytes(arr):
    buf = io.BytesIO(); tifffile.imwrite(buf, arr); return buf.getvalue()


def npy_bytes(arr):
    buf = io.BytesIO(); np.save(buf, arr); return buf.getvalue()


def fake_tile(h=128, w=128, c=12, channels_last=False):
    a = RNG.uniform(0, 10000, size=(c, h, w)).astype(np.float32)
    return np.transpose(a, (1, 2, 0)) if channels_last else a


class Base(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.tmp = tempfile.TemporaryDirectory()
        d = cls.tmp.name
        notebook_style_checkpoint(os.path.join(d, "ckpt.pt"))
        with open(os.path.join(d, "mm.json"), "w") as f:
            json.dump({"min": MIN.tolist(), "max": MAX.tolist()}, f)
        # 16 entries on purpose: the notebook saves z-stats for the 16-channel input
        with open(os.path.join(d, "z.json"), "w") as f:
            json.dump({"mean": np.linspace(0.2, 0.6, 16).tolist(), "std": np.linspace(0.2, 0.3, 16).tolist()}, f)
        cls.bundle = os.path.join(d, "bundle")
        export_bundle(os.path.join(d, "ckpt.pt"), os.path.join(d, "mm.json"), os.path.join(d, "z.json"), cls.bundle)
        cls.app = create_app(cls.bundle)
        cls.client = cls.app.test_client()
        cls.seg = cls.app.config["SEGMENTER"]

    @classmethod
    def tearDownClass(cls):
        cls.tmp.cleanup()

    def post(self, data, name="tile.tif", query="", extra=None):
        form = {"image": (io.BytesIO(data), name)}
        form.update(extra or {})
        return self.client.post("/predict" + query, data=form, content_type="multipart/form-data")


class TestExport(Base):
    def test_bundle_contents(self):
        with open(os.path.join(self.bundle, "preprocessing.json")) as f:
            cfg = json.load(f)
        self.assertEqual(cfg["in_channels"], 12)
        self.assertEqual(len(cfg["mean"]), 12)            # 16 -> first 12
        self.assertEqual(len(cfg["weights_sha256"]), 64)
        self.assertTrue(os.path.exists(os.path.join(self.bundle, cfg["weights_file"])))

    def test_rejects_16_channel_checkpoint(self):
        d = self.tmp.name
        m = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=16, classes=1)
        torch.save(m.state_dict(), os.path.join(d, "ckpt16.pt"))
        with self.assertRaises(SystemExit):
            export_bundle(os.path.join(d, "ckpt16.pt"), os.path.join(d, "mm.json"), os.path.join(d, "z.json"),
                          os.path.join(d, "bad_bundle"))


class TestPreprocessing(Base):
    def test_matches_notebook_formula(self):
        raw = fake_tile()
        raw[:, :5, :5] = -500.0                       # below train min -> must clip to 0
        raw[:, 5:9, 5:9] = 99999.0                    # above train max -> must clip to 1
        # notebook: normalize_channels() then (img - Z_MEAN[:c]) / Z_STD[:c]
        mn = MIN.astype(np.float32).reshape(-1, 1, 1); mx = MAX.astype(np.float32).reshape(-1, 1, 1)
        x = np.clip((raw - mn) / np.maximum(mx - mn, 1e-6), 0, 1)
        mean = np.linspace(0.2, 0.6, 16)[:12].astype(np.float32).reshape(-1, 1, 1)
        std = np.linspace(0.2, 0.3, 16)[:12].astype(np.float32).reshape(-1, 1, 1)
        expected = (x - mean) / std
        np.testing.assert_allclose(self.seg.preprocess(raw), expected, rtol=1e-5, atol=1e-5)

    def test_channels_last_equals_channels_first(self):
        a = fake_tile()
        p1 = self.seg.predict_proba(a)
        p2 = self.seg.predict_proba(np.transpose(a, (1, 2, 0)))
        np.testing.assert_allclose(p1, p2, atol=1e-6)

    def test_deterministic(self):
        a = fake_tile()
        np.testing.assert_array_equal(self.seg.predict(a), self.seg.predict(a))


class TestPredictEndpoint(Base):
    def test_json_response(self):
        r = self.post(tif_bytes(fake_tile()))
        self.assertEqual(r.status_code, 200)
        j = r.get_json()
        self.assertEqual((j["height"], j["width"]), (128, 128))
        mask = np.array(Image.open(io.BytesIO(base64.b64decode(j["mask_png_base64"]))))
        self.assertEqual(mask.shape, (128, 128))
        self.assertTrue(set(np.unique(mask)) <= {0, 255})
        self.assertAlmostEqual(j["water_fraction"], (mask > 0).mean(), places=6)
        self.assertEqual(j["water_pixels"], int((mask > 0).sum()))
        for k in ("rgb_png_base64", "overlay_png_base64"):
            self.assertEqual(Image.open(io.BytesIO(base64.b64decode(j[k]))).size, (128, 128))

    def test_png_response(self):
        r = self.post(tif_bytes(fake_tile()), query="?format=png")
        self.assertEqual(r.status_code, 200)
        self.assertEqual(r.mimetype, "image/png")
        img = Image.open(io.BytesIO(r.data))
        self.assertEqual((img.mode, img.size), ("L", (128, 128)))
        self.assertTrue(set(np.unique(np.array(img))) <= {0, 255})
        self.assertIn("X-Water-Fraction", r.headers)

    def test_png_matches_library_prediction(self):
        a = fake_tile()
        r = self.post(tif_bytes(a), query="?format=png")
        api_mask = (np.array(Image.open(io.BytesIO(r.data))) > 0).astype(np.uint8)
        np.testing.assert_array_equal(api_mask, self.seg.predict(a))

    def test_channels_last_tif_and_npy(self):
        a = fake_tile(channels_last=True)
        self.assertEqual(self.post(tif_bytes(a)).status_code, 200)
        self.assertEqual(self.post(npy_bytes(a), name="tile.npy").status_code, 200)

    def test_non_multiple_of_32_size_is_cropped_back(self):
        j = self.post(tif_bytes(fake_tile(100, 70))).get_json()
        self.assertEqual((j["height"], j["width"]), (100, 70))

    def test_threshold_is_applied(self):
        # Compare against the library on the same tile. (A random untrained net saturates the sigmoid, so
        # some probabilities are exactly 0.0 or 1.0 - don't assume "threshold 0 => everything is water".)
        tile = fake_tile()
        data = tif_bytes(tile)
        prob = self.seg.predict_proba(tile)
        fractions = []
        for thr in (0.1, 0.5, 0.9):
            j = self.post(data, extra={"threshold": str(thr)}).get_json()
            self.assertEqual(j["threshold"], thr)
            self.assertAlmostEqual(j["water_fraction"], float((prob > thr).mean()), places=6)
            fractions.append(j["water_fraction"])
        self.assertEqual(fractions, sorted(fractions, reverse=True))      # stricter threshold => never more water
        self.assertEqual(self.post(data, extra={"threshold": "1"}).get_json()["water_fraction"], 0.0)  # sigmoid <= 1

    def test_default_threshold_comes_from_bundle(self):
        j = self.post(tif_bytes(fake_tile())).get_json()
        self.assertEqual(j["threshold"], self.seg.threshold)

    # ---- bad input -> clean 4xx JSON errors
    def assert_error(self, resp, code):
        self.assertEqual(resp.status_code, code)
        self.assertIn("error", resp.get_json())

    def test_no_file(self):
        self.assert_error(self.client.post("/predict", data={}, content_type="multipart/form-data"), 400)

    def test_wrong_channel_count(self):
        self.assert_error(self.post(tif_bytes(fake_tile(c=11))), 400)
        self.assert_error(self.post(tif_bytes(fake_tile(c=3))), 400)

    def test_wrong_ndim(self):
        self.assert_error(self.post(npy_bytes(np.zeros((128, 128), np.float32)), name="x.npy"), 400)

    def test_unsupported_extension(self):
        self.assert_error(self.post(b"hello", name="tile.png"), 400)
        self.assert_error(self.post(b"hello", name="noext"), 400)

    def test_corrupt_files(self):
        self.assert_error(self.post(b"this is not a tiff", name="bad.tif"), 400)
        self.assert_error(self.post(b"this is not an npy", name="bad.npy"), 400)

    def test_pickled_npy_refused(self):
        buf = io.BytesIO(); np.save(buf, np.array([{"a": 1}], dtype=object), allow_pickle=True)
        self.assert_error(self.post(buf.getvalue(), name="evil.npy"), 400)

    def test_nan_rejected(self):
        a = fake_tile(); a[0, 0, 0] = np.nan
        self.assert_error(self.post(tif_bytes(a)), 400)

    def test_oversize_image_rejected(self):
        self.assert_error(self.post(npy_bytes(np.zeros((12, 1100, 40), np.float32)), name="big.npy"), 400)

    def test_bad_threshold_and_format(self):
        data = tif_bytes(fake_tile())
        self.assert_error(self.post(data, extra={"threshold": "abc"}), 400)
        self.assert_error(self.post(data, extra={"threshold": "1.5"}), 400)
        self.assert_error(self.post(data, query="?format=gif"), 400)

    def test_upload_too_large(self):
        self.app.config["MAX_CONTENT_LENGTH"] = 1000
        try:
            self.assert_error(self.post(tif_bytes(fake_tile())), 413)
        finally:
            self.app.config["MAX_CONTENT_LENGTH"] = 32 * 1024 * 1024


class TestOtherRoutes(Base):
    def test_health(self):
        j = self.client.get("/health").get_json()
        self.assertEqual((j["status"], j["in_channels"], j["encoder"]), ("ok", 12, "resnet34"))

    def test_frontend_served(self):
        r = self.client.get("/")
        self.assertEqual(r.status_code, 200)
        self.assertIn(b"Water segmentation", r.data)
        r.close()

    def test_json_404_and_405(self):
        self.assertEqual(self.client.get("/nope").status_code, 404)
        self.assertIn("error", self.client.get("/nope").get_json())
        self.assertEqual(self.client.get("/predict").status_code, 405)


if __name__ == "__main__":
    unittest.main(verbosity=2)


---
## 🧠 Step 3: Auto-detect the 3 Training Output Files & Package Model Bundle

The training pipeline produces **3 specific output files** in `week2_outputs/`:
1. `pretrained_resnet34_u_net_12ch_best.pt` (the trained model weights)
2. `channel_minmax.json` (per-channel min/max for the 12 Sentinel-2 bands)
3. `input_zscore_stats.json` (per-channel mean/std for Z-score normalization)

The cell below:
- Scans `/kaggle/input/` and `/kaggle/working/` for all 3 files.
- Automatically pairs them and runs `export_bundle()` (from `export_model.py`).
- Strictly verifies that the first convolution layer takes 12 channels and runs a forward-pass smoke test.
- Saves the verified bundle into `water_app/model_bundle/` (`water_unet_resnet34_12ch.pt` + `preprocessing.json`).
- If you haven't attached your dataset yet, it generates architecture-matched demo weights so you can test the Flask app immediately!


In [ ]:
import os
import sys
import glob
import json
import torch
import numpy as np
import segmentation_models_pytorch as smp

# ==============================================================================
# Step 3: Auto-detect the 3 Training Output Files & Export Model Bundle
# ==============================================================================
# If you attached the 3 files via Kaggle "+ Add Input", they will be auto-detected.
# You can also explicitly specify custom paths below if needed:
CUSTOM_CHECKPOINT = None   # e.g., "/kaggle/input/water-model/pretrained_resnet34_u_net_12ch_best.pt"
CUSTOM_MINMAX     = None   # e.g., "/kaggle/input/water-model/channel_minmax.json"
CUSTOM_ZSTATS     = None   # e.g., "/kaggle/input/water-model/input_zscore_stats.json"

sys.path.insert(0, APP_DIR)
from export_model import export_bundle
from inference import WaterSegmenter

BUNDLE_PATH = os.path.join(APP_DIR, "model_bundle")
os.makedirs(BUNDLE_PATH, exist_ok=True)

# Default training statistics (used as fallback if json files are not uploaded)
DEFAULT_MIN  = [-1390.0, -1110.0, -633.0, -627.0, -408.0, -335.0, -251.0, 64.0, -9999.0, 8.0, 10.0, 0.0]
DEFAULT_MAX  = [6568.0, 9659.0, 11368.0, 12041.0, 15841.0, 15252.0, 14647.0, 255.0, 4245.0, 4287.0, 100.0, 111.0]
DEFAULT_MEAN = [0.22462025406962807, 0.1491273996652211, 0.12115566004314575, 0.12616755157821738, 0.15183025795416183, 0.145941832217426, 0.10656552302612511, 0.20160297640591282, 0.7119612665860809, 0.07265980946561738, 0.27978229946416294, 0.08939620797682428]
DEFAULT_STD  = [0.03560793838468084, 0.031571210757412774, 0.0356249442769683, 0.04648571610306687, 0.06434432346007361, 0.07556986754256001, 0.06335020722213462, 0.25617821357693904, 0.10161723857259256, 0.1250486007296272, 0.22485809982091365, 0.2527648127994044]

def find_file(patterns, custom_path=None):
    if custom_path and os.path.exists(custom_path):
        return os.path.abspath(custom_path)
    for pat in patterns:
        for match in glob.glob(pat, recursive=True):
            abs_match = os.path.abspath(match)
            if not abs_match.startswith(BUNDLE_PATH) and not abs_match.endswith(".ipynb_checkpoints"):
                return abs_match
    return None

# Search for the 3 files across /kaggle/input/, /kaggle/working/, and current dir
found_ckpt = find_file([
    "/kaggle/input/**/pretrained_resnet34_u_net_12ch*.pt",
    "/kaggle/input/**/*.pt",
    "/kaggle/working/**/pretrained_resnet34_u_net_12ch*.pt",
    "week2_outputs/*.pt",
    "*.pt"
], CUSTOM_CHECKPOINT)

found_minmax = find_file([
    "/kaggle/input/**/channel_minmax*.json",
    "/kaggle/input/**/minmax*.json",
    "/kaggle/working/**/channel_minmax*.json",
    "week2_outputs/channel_minmax.json",
    "channel_minmax.json"
], CUSTOM_MINMAX)

found_zstats = find_file([
    "/kaggle/input/**/input_zscore_stats*.json",
    "/kaggle/input/**/zscore*.json",
    "/kaggle/working/**/input_zscore_stats*.json",
    "week2_outputs/input_zscore_stats.json",
    "input_zscore_stats.json"
], CUSTOM_ZSTATS)

print("=" * 70)
print("🔍 Scanning for the 3 Week 2 training output files...")
print("-" * 70)

if found_ckpt:
    print(f"  1. Model Weights:      ✅ Found: {found_ckpt}")
else:
    print("  1. Model Weights:      ⚠️ Not found (will generate demo weights)")

if found_minmax:
    print(f"  2. Channel Min/Max:    ✅ Found: {found_minmax}")
else:
    print("  2. Channel Min/Max:    ℹ️ Using pre-embedded default training stats")

if found_zstats:
    print(f"  3. Z-Score Mean/Std:   ✅ Found: {found_zstats}")
else:
    print("  3. Z-Score Mean/Std:   ℹ️ Using pre-embedded default training stats")
print("-" * 70)

# Prepare minmax and zstats file paths
temp_minmax = os.path.join(APP_DIR, "_default_minmax.json")
temp_zstats = os.path.join(APP_DIR, "_default_zstats.json")

if not found_minmax:
    with open(temp_minmax, "w") as f:
        json.dump({"min": DEFAULT_MIN, "max": DEFAULT_MAX}, f, indent=2)
    minmax_path_to_use = temp_minmax
else:
    minmax_path_to_use = found_minmax

if not found_zstats:
    with open(temp_zstats, "w") as f:
        json.dump({"mean": DEFAULT_MEAN, "std": DEFAULT_STD}, f, indent=2)
    zstats_path_to_use = temp_zstats
else:
    zstats_path_to_use = found_zstats

# Prepare weights checkpoint
if found_ckpt:
    ckpt_path_to_use = found_ckpt
else:
    print("⚠️ No .pt checkpoint found yet. Generating synthetic test weights...")
    print("   💡 Once you attach your real dataset (+ Add Input), re-run this cell!")
    demo_ckpt = os.path.join(APP_DIR, "demo_weights.pt")
    dummy_model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=12, classes=1)
    torch.save(dummy_model.state_dict(), demo_ckpt)
    ckpt_path_to_use = demo_ckpt

# Package into bundle using export_bundle()
print(f"📦 Packaging model bundle into: {BUNDLE_PATH}")
cfg = export_bundle(
    checkpoint=ckpt_path_to_use,
    minmax_path=minmax_path_to_use,
    zstats_path=zstats_path_to_use,
    out_dir=BUNDLE_PATH,
    threshold=0.5,
    encoder="resnet34"
)
print("=" * 70)

# Quick smoke test of WaterSegmenter inference
seg = WaterSegmenter(BUNDLE_PATH)
test_arr = np.random.uniform(0, 5000, size=(12, 128, 128)).astype(np.float32)
test_mask = seg.predict(test_arr)
print(f"🎉 Smoke test passed! Output mask shape: {test_mask.shape}, Device: {seg.device}")


---
## 🌐 Step 4: Launch Flask App & Expose via Cloudflare Tunnel

Kaggle notebooks run inside remote containers without open inbound ports. To access the web interface or REST API from your browser:
- We start the Flask server in the background on port `5000`.
- We use **Cloudflare Tunnel (`cloudflared`)**, which opens a secure HTTPS reverse tunnel (`https://*.trycloudflare.com`) directly to `localhost:5000`.
- No account, registration, or API key is required!


In [ ]:
# Clean up any previously running servers
!pkill -f "app.py" || true
!pkill -f "cloudflared" || true
time.sleep(1)

# 1. Download cloudflared binary if not present
if not os.path.exists("cloudflared"):
    print("⬇️ Downloading Cloudflare Tunnel (cloudflared)...")
    !curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared
    !chmod +x cloudflared

# 2. Launch Flask App in background (detached session)
flask_log = open("flask_server.log", "w")
flask_proc = subprocess.Popen(
    [sys.executable, "app.py", "--bundle", "model_bundle", "--host", "0.0.0.0", "--port", "5000"],
    cwd=APP_DIR,
    stdout=flask_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)
print(f"🚀 Flask server started (PID: {flask_proc.pid})")

# 3. Wait for Flask health check
print("⏳ Waiting for Flask server to initialize...")
server_ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:5000/health", timeout=1) as resp:
            if resp.status == 200:
                server_ready = True
                print("✅ Flask server is responding on http://127.0.0.1:5000/health!")
                break
    except Exception:
        time.sleep(0.5)

if not server_ready:
    print("❌ Flask server failed to start. Last log lines:")
    with open("flask_server.log") as f:
        print(f.read())
    raise RuntimeError("Flask initialization failed.")

# 4. Start Cloudflare Tunnel (detached session)
import re
tunnel_log = open("tunnel.log", "w")
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:5000"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

public_url = None
print("⏳ Establishing secure Cloudflare tunnel...")
for _ in range(40):
    time.sleep(0.5)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as f:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if match:
                public_url = match.group(0)
                break

if public_url:
    from IPython.display import display, HTML
    display(HTML(f'''
    <div style="background-color: #f0fdf4; border: 2px solid #22c55e; border-radius: 12px; padding: 20px; font-family: sans-serif; margin: 15px 0;">
        <h3 style="color: #15803d; margin: 0 0 10px 0;">🎉 Water Segmentation App is Live!</h3>
        <p style="margin: 0 0 14px 0; color: #166534; font-size: 1.05em;">Your web application and REST API are accessible publicly over HTTPS:</p>
        <div style="margin-bottom: 14px;">
            <a href="{public_url}" target="_blank" style="background-color: #22c55e; color: white; padding: 12px 24px; border-radius: 8px; text-decoration: none; font-weight: bold; font-size: 1.1em; display: inline-block;">
                🌐 Open Water Segmentation Web UI ↗
            </a>
        </div>
        <div style="background: white; border: 1px solid #bbf7d0; border-radius: 8px; padding: 12px; font-size: 0.95em; color: #374151;">
            <b>Public URL:</b> <a href="{public_url}" target="_blank">{public_url}</a><br>
            <b>Health Endpoint:</b> <a href="{public_url}/health" target="_blank">{public_url}/health</a><br>
            <b>Predict Endpoint:</b> <code>{public_url}/predict</code> (POST multipart/form-data)
        </div>
    </div>
    '''))
else:
    print("⚠️ Could not extract public URL from tunnel.log. Recent log:")
    with open("tunnel.log") as f:
        print(f.read())


---
## 🧪 Step 5: Test the API directly in Python & Visualize Results

Now we test the running Flask app by simulating a realistic 12-channel Sentinel-2 tile with simulated water bodies (using band absorption signatures). We call both endpoints:
- `POST /predict` (JSON response with metrics & base64 preview images)
- `POST /predict?format=png` (Raw binary 8-bit PNG mask)


In [ ]:
import io
import base64
import requests
import tifffile
from PIL import Image
import matplotlib.pyplot as plt

# 1. Health check verification
health = requests.get("http://127.0.0.1:5000/health").json()
print("🩺 Health Check Status:")
print(json.dumps(health, indent=2))

# 2. Synthesize a 12-channel test tile (128x128 pixels)
# Bands: Coastal, Blue, Green, Red, NIR, SWIR1, SWIR2, QA, Merit DEM, Copernicus DEM, WorldCover, Water Occ
np.random.seed(42)
test_tile = np.random.uniform(500, 2500, size=(12, 128, 128)).astype(np.float32)

# Simulate a central water lake:
# Deep absorption in NIR/SWIR bands (4, 5, 6) and high water occurrence prob (band 11)
test_tile[4:7, 30:95, 30:95] *= 0.1
test_tile[11, 30:95, 30:95] = 95.0

# Encode to GeoTIFF bytes
tif_buf = io.BytesIO()
tifffile.imwrite(tif_buf, test_tile)
tif_bytes = tif_buf.getvalue()

# Save sample file to disk for user download/testing
with open("sample_12band_tile.tif", "wb") as f:
    f.write(tif_bytes)
print(f"\n📁 Saved sample 12-channel tile to: sample_12band_tile.tif ({len(tif_bytes)/1024:.1f} KB)")

# 3. Send prediction request to local Flask server
response = requests.post(
    "http://127.0.0.1:5000/predict",
    files={"image": ("sample_12band_tile.tif", tif_bytes, "image/tiff")},
    data={"threshold": 0.5}
)
res = response.json()

print(f"\n📊 Inference Results:")
print(f"  Status Code:    {response.status_code}")
print(f"  Dimensions:     {res['width']} x {res['height']} pixels")
print(f"  Inference Time: {res['inference_ms']} ms")
print(f"  Water Fraction: {res['water_fraction']*100:.2f}% ({res['water_pixels']} pixels)")

# 4. Decode base64 PNGs and plot side-by-side
rgb_img = Image.open(io.BytesIO(base64.b64decode(res['rgb_png_base64'])))
mask_img = Image.open(io.BytesIO(base64.b64decode(res['mask_png_base64'])))
overlay_img = Image.open(io.BytesIO(base64.b64decode(res['overlay_png_base64'])))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rgb_img)
axes[0].set_title("True Colour (Bands 4, 3, 2)", fontsize=13, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(mask_img, cmap="gray")
axes[1].set_title(f"Predicted Mask ({res['water_fraction']*100:.1f}% water)", fontsize=13, fontweight="bold")
axes[1].axis("off")

axes[2].imshow(overlay_img)
axes[2].set_title("Water Overlay (Blue)", fontsize=13, fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()

# 5. Test raw PNG format endpoint
png_resp = requests.post(
    "http://127.0.0.1:5000/predict?format=png",
    files={"image": ("sample_12band_tile.tif", tif_bytes, "image/tiff")}
)
print("🖼️ PNG Endpoint Headers:")
print(f"  Content-Type:       {png_resp.headers.get('Content-Type')}")
print(f"  X-Water-Fraction:   {png_resp.headers.get('X-Water-Fraction')}")
print(f"  X-Threshold:        {png_resp.headers.get('X-Threshold')}")
print(f"  Payload size:       {len(png_resp.content)} bytes")


---
## 🖥️ Step 6: Interactive Web UI directly in Kaggle

You can interact with the web interface right inside the notebook using this embedded iframe!
*(Tip: You can download `sample_12band_tile.tif` from Kaggle output and drag-and-drop it into the box below)*


In [ ]:
from IPython.display import IFrame, display, HTML

if "public_url" in globals() and public_url:
    display(HTML(f'''
    <p>If the frame below is blocked by browser third-party restrictions, open directly: 
    <a href="{public_url}" target="_blank" style="font-weight:bold; color:#2563eb;">Open in New Tab ↗</a></p>
    '''))
    display(IFrame(src=public_url, width="100%", height=800))
else:
    print("Public URL is not defined. Please run Step 4 first.")


---
## 🧪 Step 7: Automated Unit Tests

Run the full 25-test suite (`water_app/tests/test_api.py`) covering:
- Checkpoint export & channel validation
- Normalization and clipping formulas matching the training notebook
- Endpoint status codes, error handling (400, 413, 404, 405)
- Output formats (`json` and `png`) and threshold effects


In [ ]:
# Run all 25 automated unit tests
!python -m unittest discover -s water_app/tests -v


---
## 🛠️ Step 8: Server Management & Log Viewer

Useful functions to monitor server traffic or stop the server when you are done.


In [ ]:
def show_flask_logs(lines=25):
    """Display the latest Flask application logs."""
    if os.path.exists("flask_server.log"):
        with open("flask_server.log") as f:
            content = f.readlines()
            print(f"--- Flask Server Logs (Last {min(lines, len(content))} lines) ---")
            print("".join(content[-lines:]))
    else:
        print("No flask_server.log found.")

def show_tunnel_logs(lines=20):
    """Display the latest Cloudflare tunnel logs."""
    if os.path.exists("tunnel.log"):
        with open("tunnel.log") as f:
            content = f.readlines()
            print(f"--- Cloudflare Tunnel Logs (Last {min(lines, len(content))} lines) ---")
            print("".join(content[-lines:]))
    else:
        print("No tunnel.log found.")

def stop_server():
    """Stop Flask server and tunnel background processes."""
    !pkill -f "app.py" || true
    !pkill -f "cloudflared" || true
    print("🛑 Flask server and Cloudflare tunnel stopped.")

# Show recent server activity
show_flask_logs(15)


---
## 🔌 Step 9 (Optional): Alternative Tunneling via ngrok

If you prefer using [ngrok](https://ngrok.com) instead of Cloudflare Tunnel, you can run this cell with your ngrok authentication token:
```python
from pyngrok import ngrok

# Insert your ngrok token here
NGROK_TOKEN = "YOUR_NGROK_AUTH_TOKEN"
ngrok.set_auth_token(NGROK_TOKEN)

# Connect to local port 5000
ngrok_tunnel = ngrok.connect(5000)
print(f"ngrok Public URL: {ngrok_tunnel.public_url}")
```
